# Week 12: Harness Setup & Comparison Lab

> ZoroLogistics ETA CLI · set up the harnesses, write the spec, build agentically, then compare the harness that built it.

# Requirements: none beyond shell tools (install per reference/skills/ guides)


## 0. What this lab is for

This notebook is the Monday to Thursday spine of Week 12. It is deliberately unexciting: it
checks which harnesses you have, hands you the SPEC.md template, and gives you a place to
record evidence while you build the same CLI in two harnesses. The Python cells run with
nothing but your Week-1 environment (numpy, pandas) and the local `zoro` package, no API
keys, no GPU.

**The Monday-to-Friday map:**

- **Mon**: run the tool check below; install + authenticate two harnesses (see
  `reference/skills/claude-code.md`, `reference/skills/opencode.md`, `reference/skills/deepseek-harness.md`, `reference/skills/cursor.md`).
- **Tue**: copy the SPEC.md template into your fork's `specs/` folder and fill it in.
- **Wed/Thu**: drive harness A to build the CLI, run its tests, then repeat in harness B;
  record plans, tokens, cost, and quality in the worksheet.
- **Fri**: the final cell prints your harness-readiness score (a number); pair it with the
  1-page comparison write-up.


## 1. Tool check: what is installed?

The four tools below matter this week. The check is **graceful**: a missing tool prints
`not installed , see reference/skills/…` instead of failing the notebook, so you can read the whole
lab even on a machine with nothing installed yet.


In [ ]:
%%bash
echo '== Claude Code (Anthropic) =='
claude --version 2>/dev/null || echo 'not installed, see reference/skills/claude-code.md'

echo '== OpenCode (SST) =='
opencode --version 2>/dev/null || echo 'not installed, see reference/skills/opencode.md'

echo '== DeepSeek Harness / DSH (DeepSeek) =='
dsh --version 2>/dev/null || echo 'not installed, see reference/skills/deepseek-harness.md'

echo '== Ollama (optional local model backend) =='
ollama --version 2>/dev/null || echo 'not installed, see reference/skills/ (Week 8 local-stack guide)'


In [ ]:
import shutil

# The three coding harnesses this week targets, plus the optional local backend.
TOOLS = {
    'claude':   'reference/skills/claude-code.md',
    'opencode': 'reference/skills/opencode.md',
    'dsh':      'reference/skills/deepseek-harness.md',
    'ollama':   'reference/skills/ (Week 8 local-stack guide)',
}

installed = {name: shutil.which(name) is not None for name in TOOLS}
for name, ok in installed.items():
    print(name, '->', 'installed' if ok else 'missing, see ' + TOOLS[name])

# Harness-readiness = number of the three CODING harnesses present (claude/opencode/dsh).
harness_readiness = sum(1 for n in ('claude', 'opencode', 'dsh') if installed[n])
print('harness_readiness (0-3) =', harness_readiness)


## 2. The Week-12 SPEC.md template

A **spec is the definition of done the loop closes against**, not documentation for humans
(see `reference/knowledge-base/01-ai-engineering-discipline.md`). Copy the block below into
`specs/eta-cli-SPEC.md` in your fork and fill every bracket. The single most important line
is the **refused tradeoff**: it is the one place you tell the agent, in advance, where it
is NOT allowed to resolve ambiguity by guessing.


```markdown
# SPEC.md: ZoroLogistics ETA CLI

## User
A ZoroLogistics operations analyst who has a shipment id and wants, in one terminal
command, the planned vs. actual arrival and the delay, without opening a notebook
or touching a database.

## What it does
`eta-cli S0000001` prints, to stdout:
  shipment_id, carrier_id, lane_id, commodity, planned_arrival, actual_arrival,
  delay_hours, is_on_time
Reads from the shipped data/shipments.csv (generated in Week 1 with seed 42).

## Constraints
- Python 3.12, standard library + pandas only; no network, no API keys.
- Deterministic: same input → same output; no randomness.
- `--json` flag emits the same fields as one JSON object.
- Unknown shipment id → exit code 2 and a message on stderr (never a fabricated row).

## Refused tradeoff (the one we will NOT make)
We refuse to trade correctness for a clean exit: the CLI must never invent an ETA.
If the id is absent or the row has NaN arrival data, it exits non-zero and says so
instead of printing a made-up number. A confident wrong ETA is worse than an error.

## Test plan (the verifier)
1. Lookup a known shipment → correct row fields, exit code 0.
2. Lookup an unknown id → exit code 2, stderr message, no stdout row.
3. `--json` → valid JSON with exactly the eight fields above.
4. NaN arrival row → non-zero exit, no fabricated ETA.
5. Empty/missing CSV → non-zero exit, clear message.
6. `--help` → usage text, exit code 0.
```


## 3. The comparison worksheet

Run the **same** task (the ETA CLI, from the same SPEC.md) in two harnesses. Record three
things for each: **plan** (did it lay out a plan? how close to your spec?), **cost**
(tokens / dollars), and **quality** (did its tests pass first try? how clean was the
diff?). A one-line *it was fine* is not evidence, fill every cell.


In [ ]:
import pandas as pd

# Fill this in as you run the build in each harness. Leave None until then.
worksheet = pd.DataFrame([
    {
        'harness': 'Harness A (e.g. claude)',
        'task': 'ETA CLI from SPEC.md',
        'plan_quality_1to5': None,
        'tokens_used': None,
        'cost_usd': None,
        'tests_green_first_try': None,
        'diff_files_changed': None,
        'code_quality_1to5': None,
        'notes': '',
    },
    {
        'harness': 'Harness B (e.g. opencode/dsh)',
        'task': 'ETA CLI from SPEC.md',
        'plan_quality_1to5': None,
        'tokens_used': None,
        'cost_usd': None,
        'tests_green_first_try': None,
        'diff_files_changed': None,
        'code_quality_1to5': None,
        'notes': '',
    },
])
worksheet


## 4. Cost-log helper

Token and dollar numbers are the only honest way to compare *free* harnesses (they all cost
your model provider). Append one row per run; the helper totals tokens and dollars so the
Friday write-up can quote a single per-build cost instead of a vibe.


In [ ]:
import pandas as pd
from datetime import datetime, timezone

# The cost log: one row per agent run. You fill tokens + dollars from your provider
# usage page; the helper does the arithmetic so Friday can quote totals.
cost_log = []

def log_cost(harness, task, tokens_in, tokens_out, cost_usd, note=''):
    cost_log.append({
        'ts_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'harness': harness,
        'task': task,
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'tokens_total': tokens_in + tokens_out,
        'cost_usd': cost_usd,
        'note': note,
    })
    return cost_log

log_cost('claude', 'ETA CLI build', 0, 0, 0.0, 'example, replace with real numbers')
log_cost('opencode', 'ETA CLI build (2nd harness)', 0, 0, 0.0, 'example, replace')

cost_frame = pd.DataFrame(cost_log)
print('runs:', len(cost_frame), '| total_tokens:', cost_frame['tokens_total'].sum(), '| total_cost_usd:', round(cost_frame['cost_usd'].sum(), 4))
cost_frame


In [ ]:
import sys, pathlib
# zoro import cell (per week template)
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
# Fallback: locate the repo root by finding the zoro/ package, in case the notebook
# was launched from a subdirectory.
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'zoro').is_dir():
        sys.path.insert(0, str(_c))
        break

from zoro import data  # noqa: E402

# The CLI your agent builds reads this generator's output. Recreate a tiny
# deterministic slice so you can eyeball the columns the SPEC.md references.
_ship = data.shipments(n=200, seed=42)
print(_ship[['shipment_id', 'carrier_id', 'lane_id', 'commodity',
             'planned_arrival', 'actual_arrival', 'delay_hours', 'is_on_time']].head(5))
print('shipments columns:', list(_ship.columns))


## 5. Final metric: harness readiness

The number below is your **harness-readiness score**: how many of the three coding
harnesses (Claude Code, OpenCode, DSH) are installed and on your PATH. It is the single
number this notebook reports. Record it, then pair it with the filled worksheet and the
1-page comparison for Friday.


In [ ]:
import shutil

# Final metric: harness-readiness (0-3). Recomputed so this cell stands alone.
final_readiness = sum(1 for n in ('claude', 'opencode', 'dsh') if shutil.which(n))
print(final_readiness)
